## Приоритезация обращений. Model

**Подход к решению**:
1) провел EDA (см. файл `eda.ipynb` - там есть все выводы)
2) валидация скользящим expanding-окном по дням, каждый фолд разделен на train/val/test; val (2 дня) - для подбора параметров, test (3 дня) - для оценки моделей;
3) модель - Catboost, параметры подбирал с optuna;
4) [TODO:] с помощью `events.csv` провел feature engineering для добавления новых признаков

In [130]:
# подхватывать правки локальных модулей без рестарта ядра
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# локальные модули
import preprocessor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [131]:
ROOT = Path(".")
DATA_DIR = ROOT / "data"

TARGET = "target"
RANDOM_STATE = 42

TEST_DAYS = 3  # test
VALID_DAYS = 2  # val
N_FOLDS = 3  # число фолдов
MIN_TRAIN_DAYS = 5  # минимальный размер train без val

SEEDS = (0, 1, 2)  # для усреднения по сидам
TUNE_SEEDS = (0, 1)

### Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.

In [132]:
train = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test_df:", test_df.shape)
print("events:", events.shape)

train: (13694, 119)
test_df: (4306, 118)
events: (254705, 7)


### Выбор признаков TODO:

### Валидация

`test_df` лежит строго после `train` по времени:
- `train 07.04-22.04 = 16 дней`
- `test_df 23.04-27.04 = 5 дней`

#### Метрики

In [133]:
def daily_ap(y_true, scores, dates) -> float:
    """Метрика соревнования: AP считается внутри каждого дня и усредняется по дням.

    Дни без единого положительного примера пропускаем - AP на них не определён.
    """
    df = pd.DataFrame({"y": y_true, "s": scores, "d": dates})
    per_day = [average_precision_score(g["y"], g["s"])
               for _, g in df.groupby("d") if g["y"].sum() > 0]
    return float(np.mean(per_day))

#### Схема

Обычная кросс-валидация не подходит. Двигаемся скользящим окном по дням, чтобы не было data leak. **Минимальный квант --- 1 день**, т.к. метрика Daily AP.

Фолд --- это **тройка подряд идущих блоков** `train | valid | test`, которая едет от конца выборки к началу. `valid` стоит вплотную перед `test`: на нём подбираются гиперпараметры этого же фолда, и подбор всегда смотрит на дни, непосредственно предшествующие замеру, --- ровно так же, как перед сабмитом.

```
fold 0: train 07.04-17.04 (11д) | valid 18-19 | test 20-22
fold 1: train 07.04-14.04  (8д) | valid 15-16 | test 17-19
fold 2: train 07.04-11.04  (5д) | valid 12-13 | test 14-16
```

Шаг тройки равен `TEST_DAYS`, поэтому **тест-блоки не перекрываются**: три замера сделаны на разных днях и их можно усреднять. val-блоки тоже не пересекаются с тестами своего фолда => параметры подбираются честно.

**Expanding, а не rolling**: данных мало, роскоши дропать начало выборки нет => абсолютные скоры фолдов между собой несравнимы, поэтому решения принимаем по разностям моделей внутри фолда, а не по разнице средних.

In [134]:
def folds(df, valid_days=VALID_DAYS, test_days=TEST_DAYS, n_folds=N_FOLDS, min_train_days=MIN_TRAIN_DAYS, step_days=None):
    """
    Тройки (train, valid, test) из подряд идущих дней, едущие к началу выборки.
    
    - step_days=None ---> шаг равен длине тест-блока, то есть тесты не перекрываются.
    """
    step_days = step_days or test_days
    days = pd.to_datetime(df["assignment_date"]).dt.date
    ordered = sorted(days.unique())

    for k in range(n_folds):
        test_end = len(ordered) - k * step_days
        test_start = test_end - test_days
        valid_start = test_start - valid_days

        if valid_start < min_train_days:
            raise ValueError(f"фолд {k} не помещается: на train осталось бы {valid_start}д "
                             f"при минимуме {min_train_days}")

        yield (df[days < ordered[valid_start]],
               df[days.isin(ordered[valid_start:test_start])],
               df[days.isin(ordered[test_start:test_end])])


show = lambda df: sorted(pd.to_datetime(df.assignment_date).dt.date.unique())

for i, (tr, val, te) in enumerate(folds(train)):
    total = len(tr) + len(val) + len(te)
    part = lambda df, label: (f"{label} {show(df)[0]}..{show(df)[-1]} "
                              f"({len(show(df))}д, {len(df)} строк, {len(df) / total:.0%})")
    print(f"fold {i}: {part(tr, 'train')} | {part(val, 'valid')} | {part(te, 'test')}")

fold 0: train 2026-04-07..2026-04-17 (11д, 9397 строк, 69%) | valid 2026-04-18..2026-04-19 (2д, 1738 строк, 13%) | test 2026-04-20..2026-04-22 (3д, 2559 строк, 19%)
fold 1: train 2026-04-07..2026-04-14 (8д, 6743 строк, 61%) | valid 2026-04-15..2026-04-16 (2д, 1753 строк, 16%) | test 2026-04-17..2026-04-19 (3д, 2639 строк, 24%)
fold 2: train 2026-04-07..2026-04-11 (5д, 4255 строк, 50%) | valid 2026-04-12..2026-04-13 (2д, 1646 строк, 19%) | test 2026-04-14..2026-04-16 (3д, 2595 строк, 31%)


[TODO:идея] возможно есть смысл пересчитать запуски `folds` для разные значения размеров valid, test и усреднить

### Модель

#### Модель (#0.Baseline LogReg + #1. LogReg)

Используем простую Logistic Regression:

- числовые признаки: заполнение пропусков медианой и scaling;
- категориальные признаки: заполнение самым частым значением и one-hot encoding;
- `class_weight="balanced"` из-за дисбаланса классов.

`baseline` без изменений из `quickstart.ipynb`, `logreg` имеет препроцессинг признаков 

In [135]:
def build_baseline(seed=RANDOM_STATE, **params):
    """Логрег детерминирован, сид на него не влияет - но интерфейс общий с бустингом."""
    return preprocessor.Pipeline(
        events=events,
        baseline=True,
        model=LogisticRegression(random_state=seed, max_iter=1000,
                                 class_weight="balanced", **params),
    )

In [136]:
def build_logreg(seed=RANDOM_STATE, **params):
    """Логрег детерминирован, сид на него не влияет - но интерфейс общий с бустингом."""
    return preprocessor.Pipeline(
        events=events,
        model=LogisticRegression(random_state=seed, max_iter=1000,
                                 class_weight="balanced", **params),
    )

#### Модель (#2. Catboost)

Логрегрессии нужен полный препроцессинг, бустингу он наоборот вредит. Поэтому CatBoost собираем с `sklearn_preprocess=False`, то есть без `ColumnTransformer`:

- **пропуски не импутирую** --- CatBoost обрабатывает `NaN` нативно, а замена медианой стёрла бы саму информацию о пропуске;
- **не делаю One-Hot** --- вместо него встроенная обработка категориальных признаков;
- **`boosting_type="Plain"`** вместо `Ordered`, который CatBoost включает сам на малых выборках --- быстрее и не хуже

In [137]:
def build_catboost(seed=RANDOM_STATE, **params):
    """CatBoost без sklearn-препроцессинга. Пустой params => дефолтная конфигурация."""
    return preprocessor.Pipeline(
        events=events,
        sklearn_preprocess=False,
        model=CatBoostClassifier(
            random_seed=seed,
            iterations=3000,
            early_stopping_rounds=100,
            auto_class_weights="Balanced",
            cat_features=preprocessor.CATEGORICAL_COLUMNS,  # NaN в категориальных нет
            boosting_type="Plain",
            verbose=False,
            **params,
        ),
    )

#### Модель (#3. Бэггинг Catboost + LightGBM/XGBoost)

**Идея**: данные фреймворки по разному строят деревья при реализации GB => их аггрегация может дать лучший результат

TODO:

#### Подбор гиперпараметров (optuna + Catboost)

Подбор только для `CatBoost`. 

TPE, а не сетка: пространство непрерывное.

`tune` принимает `train` и `valid` **одного фолда**, поэтому у каждого фолда набор параметров свой + усредняем значение по сидам.

In [138]:
def fit_score(build_model, train_df, test_df, seeds=SEEDS, val_df=None) -> list:
    """
    В модель передаём строки целиком, а не срез по признакам: препроцессору нужны
    lead_id и assignment_ts, чтобы приджойнить события до момента назначения.
    
    val_df нужен только для CatBoost для early_stopping
    """
    scores = []
    for seed in seeds:
        model = build_model(seed).fit(train_df, train_df[TARGET], eval_set=(val_df, val_df[TARGET]) if val_df is not None else None)
        proba = model.predict_proba(test_df)[:, 1]
        scores.append(daily_ap(test_df[TARGET], proba, test_df["assignment_date"]))
    return scores

In [139]:
N_TRIALS = 20
optuna.logging.set_verbosity(optuna.logging.WARNING)

def suggest_params(trial) -> dict:
    return {
        # "iterations": trial.suggest_int("iterations", 200, 800, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        # "random_strength": trial.suggest_float("random_strength", 0.5, 5.0),
    }

def tune(train_part, valid_part, n_trials=N_TRIALS, seeds=TUNE_SEEDS):
    """
    optuna на valid-блоке одного фолда. Возвращает study целиком: best_params
    идут в замер, а разброс лучших конфигураций между фолдами - в диагностику
    
    на valid части обучаются одновременно и параметры, и early_stopping,
    чтобы не дообучаться на valid перед замером
    """
    def objective(trial):
        params = suggest_params(trial)
        build = lambda seed: build_catboost(seed, **params)
        return float(np.mean(fit_score(build_model=build, train_df=train_part, test_df=valid_part, val_df=valid_part, seeds=seeds)))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=n_trials)
    return study

#### Сравнение результатов моделей

Один проход по фолдам: на каждом подбираем параметры для кэтбуста на его `valid`, обучаемся на `train` (`valid` для CatBoost остаётся честным eval_set под early stopping, поэтому в обучение не попадает) и меряем все модели на его `test`.

Ориентир из бейзлайна (`make_validation_split` в `quickstart.ipynb`, обычный split 80/20 по датам): AP **0.48506**. Напрямую не сравнить --- другая схема валидации, но совпадает `baseline` модель с `quickstart.ipynb`.

In [ ]:
def walk_forward(df, n_trials=N_TRIALS) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows, chosen = [], {}

    for k, (tr, val, te) in enumerate(folds(df)):
        study = tune(tr, val, n_trials=n_trials)
        chosen[f"fold {k}"] = study.best_params

        # дообучаться на valid перед замером нельзя - т.к. val передается в eval_set
        # train_df = pd.concat([tr, val])
        builders = {
            "baseline": build_baseline,
            "logreg": build_logreg,
            "catboost_default": build_catboost,
            "catboost_tuned": lambda seed: build_catboost(seed, **study.best_params),
        }
        for name, build in builders.items():
            # передаем валидац. выборку только в Catboost
            val_kwarg = val if name.startswith("catboost") else None
            for score in fit_score(build_model=build, train_df=tr, test_df=te, val_df=val_kwarg):
                rows.append({"fold": k, "model": name, "daily_ap": score})

        print(f"fold {k}: test {show(te)[0]}..{show(te)[-1]}  "
              f"на valid best={study.best_value:.4f}")

    return pd.DataFrame(rows), pd.DataFrame(chosen).T

In [141]:
results, best_params = walk_forward(train)

per_fold = results.groupby(["fold", "model"])["daily_ap"].mean().unstack()
display(per_fold.round(4))
display(per_fold.agg(["mean", "std"]).round(4))

fold 0: test 2026-04-20..2026-04-22  на valid best=0.6356
fold 1: test 2026-04-17..2026-04-19  на valid best=0.5689
fold 2: test 2026-04-14..2026-04-16  на valid best=0.5923


model,baseline,catboost_default,catboost_tuned,logreg
fold,,,,
0,1.0,0.5943,0.6091,0.5465
1,1.0,0.6118,0.6098,0.5334
2,1.0,0.5351,0.5477,0.4846


model,baseline,catboost_default,catboost_tuned,logreg
mean,1.0,0.5804,0.5889,0.5215
std,0.0,0.0402,0.0357,0.0326


выше писал, что смотрим на разницу по каждому фолду

In [142]:
delta = per_fold["catboost_tuned"] - per_fold["catboost_default"]
noise = results.groupby(["fold", "model"])["daily_ap"].std().max()

print(f"tuned - default по фолдам: {', '.join(f'{d:+.4f}' for d in delta)}")
print(f"среднее {delta.mean():+.4f}   разброс по сидам внутри фолда, макс sd = {noise:.4f}")

tuned - default по фолдам: +0.0147, -0.0020, +0.0126
среднее +0.0085   разброс по сидам внутри фолда, макс sd = 0.0085


### Submission

Обучаем модель на всем train и строим score для test_df.
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

In [143]:
FINAL_MODEL = "catboost_tuned"
final_params = best_params.loc["fold 0"].to_dict()
builders = {
    "baseline": build_baseline,
    "logreg": build_logreg,
    "catboost_default": build_catboost,
    "catboost_tuned": lambda seed: build_catboost(seed, **final_params),
}
final_params

{'learning_rate': 0.04578297540276922,
 'depth': 4.0,
 'l2_leaf_reg': 17.061837680423547}

In [144]:
final_model = builders[FINAL_MODEL](RANDOM_STATE).fit(train, train[TARGET])

In [145]:
submission = pd.DataFrame({
    "lead_id": test_df["lead_id"].astype(str),
    "score": final_model.predict_proba(test_df)[:, 1],
})
submission.to_csv(ROOT / "submission.csv", index=False)
submission.head()

,lead_id,score
0,lead_97e409eb8f8c8246,0.014650
1,lead_55310edb4489f9e9,0.414839
2,lead_e7f653a2c6a7eee8,0.801701
3,lead_22f8e1cfc487ac20,0.037471
4,lead_48b638b839abfac3,0.091552


In [146]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test_df)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")

submission.csv is ready
